<div dir="rtl">

# منصة تحويل الكتب العربية إلى Markdown

شغّل الخلايا الثلاث بالترتيب (زر ▶)، وستحصل على **رابط للمنصة** تفتحه من هاتفك.

1. **التثبيت** — عدة دقائق، مرة واحدة لكل جلسة.
2. **ربط Drive** (اختياري لكن مُستحسن) — لحفظ النتائج في Drive.
3. **تشغيل المنصة** — يطبع رابطًا اضغطه: ترفع PDF، تتابع التقدم، تقرأ الكتاب، تنزّل الناتج.

⚠ **مهم:**
- اترك تبويب Colab مفتوحًا أثناء العمل — إغلاقه يوقف المنصة.
- الرابط يتغيّر في كل تشغيل جديد.
- المعالجة على خوادم Google لا على جهازك.

</div>

In [ ]:
#@title ١) التثبيت { display-mode: "form" }
%pip install -q "paddlepaddle==3.3.1" "paddleocr[doc-parser]==3.7.0" pymupdf fastapi uvicorn python-multipart markdown
!rm -rf /content/book_ocr && git clone -q https://github.com/7aidaraa/book_ocr /content/book_ocr
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared
print("\u2713 التثبيت اكتمل — شغّل الخلية التالية")

In [ ]:
#@title ٢) ربط Google Drive لحفظ النتائج (اختياري) { display-mode: "form" }
import os
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive", force_remount=False)

# نجعل مجلد النتائج داخل المنصة يشير إلى Drive مباشرة
target = Path("/content/drive/MyDrive/كتب-محوّلة")
target.mkdir(parents=True, exist_ok=True)

output_link = Path("/content/book_ocr/data/output")
output_link.parent.mkdir(parents=True, exist_ok=True)
if output_link.is_symlink() or output_link.exists():
    if output_link.is_symlink():
        output_link.unlink()
    else:
        import shutil; shutil.rmtree(output_link)
os.symlink(target, output_link)

print(f"\u2713 النتائج ستُحفظ في Drive داخل: كتب-محوّلة/")

In [ ]:
#@title ٣) تشغيل المنصة والحصول على الرابط { display-mode: "form" }
#@markdown **لرابط ثابت لا يتغيّر (اختياري — إعداد مرة واحدة):** أنشئ حسابًا مجانيًا في ngrok.com،
#@markdown انسخ الـAuthtoken، واحجز نطاقك المجاني من Domains، والصقهما هنا.
#@markdown يُحفظان في Drive تلقائيًا فلا تعيد إدخالهما. اتركهما فارغين = رابط مؤقت متغيّر.
ngrok_token = ""  #@param {type:"string"}
ngrok_domain = ""  #@param {type:"string"}

import os, re, subprocess, sys, time, urllib.request
from pathlib import Path

os.chdir("/content/book_ocr")

# استرجاع/حفظ إعدادات الرابط الثابت في Drive
cfg = Path("/content/drive/MyDrive/كتب-محوّلة/ngrok.txt")
ngrok_token, ngrok_domain = ngrok_token.strip(), ngrok_domain.strip()
if (not ngrok_token or not ngrok_domain) and cfg.exists():
    saved = dict(l.split("=", 1) for l in cfg.read_text().splitlines() if "=" in l)
    ngrok_token = ngrok_token or saved.get("token", "").strip()
    ngrok_domain = ngrok_domain or saved.get("domain", "").strip()
if ngrok_token and ngrok_domain and cfg.parent.exists():
    cfg.write_text(f"token={ngrok_token}\ndomain={ngrok_domain}\n")

server = subprocess.Popen(
    [sys.executable, "run.py"],
    env={**os.environ, "HOST": "0.0.0.0", "PORT": "8000"},
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)

print("تشغيل الخادم...")
for _ in range(120):
    try:
        urllib.request.urlopen("http://127.0.0.1:8000/", timeout=2)
        break
    except Exception:
        if server.poll() is not None:
            raise SystemExit("✗ فشل تشغيل الخادم:\n" + server.stdout.read())
        time.sleep(1)
else:
    raise SystemExit("✗ الخادم لم يستجب")

public_url = None
if ngrok_token and ngrok_domain:
    print("فتح الرابط الثابت عبر ngrok...")
    if not Path("/usr/local/bin/ngrok").exists():
        subprocess.run(
            "wget -q https://bin.equinox.io/c/bNyj1mQVY4c/ngrok-v3-stable-linux-amd64.tgz -O /tmp/ngrok.tgz"
            " && tar -xzf /tmp/ngrok.tgz -C /usr/local/bin", shell=True, check=True)
    subprocess.run(["ngrok", "config", "add-authtoken", ngrok_token],
                   check=True, capture_output=True)
    tunnel = subprocess.Popen(
        ["ngrok", "http", "--url", ngrok_domain, "8000", "--log", "stdout"],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    time.sleep(5)
    if tunnel.poll() is not None:
        print("✗ فشل ngrok — تحقق من الرمز والنطاق. سجله:")
        print(tunnel.stdout.read()[:1500])
    else:
        public_url = f"https://{ngrok_domain.removeprefix('https://')}"

if not public_url:
    print("فتح رابط مؤقت عبر cloudflared...")
    tunnel = subprocess.Popen(
        ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000", "--no-autoupdate"],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    deadline = time.time() + 90
    while time.time() < deadline:
        line = tunnel.stdout.readline()
        if not line and tunnel.poll() is not None:
            break
        match = re.search(r"https://[-\w]+\.trycloudflare\.com", line)
        if match:
            public_url = match.group(0)
            break

if not public_url:
    raise SystemExit("✗ تعذر إنشاء الرابط — أعد تشغيل هذه الخلية")

print("\n" + "=" * 52)
print("  ✓ المنصة تعمل — افتح هذا الرابط:")
print(f"  {public_url}")
print("=" * 52)
print("\n⚠ اترك هذه الخلية تعمل ولا تغلق التبويب.")
print("   لإيقاف المنصة: اضغط زر التوقف ■ في هذه الخلية.\n")
print("--- سجل الخادم ---")
for line in server.stdout:
    print(line, end="")